# Notebook A: Private AI Container Embeddings

This notebook demonstrates vector search using Oracle Private AI Services Container (`privateai`) for embedding generation.

Flow:
- call Private AI REST APIs (`/health`, `/v1/models`, `/v1/embeddings`)
- store vectors in Oracle AI Database
- run cosine similarity search


In [ ]:
import os
import json
import requests
import oracledb
from dotenv import dotenv_values


## 1) Load configuration from `/home/.env`

In [ ]:
ENV_PATH = os.getenv('LAB_ENV_FILE', '/home/.env')
env = dotenv_values(ENV_PATH) if os.path.exists(ENV_PATH) else {}

DB_USER = os.getenv('DB_USER') or env.get('USERNAME') or 'ADMIN'
DB_PASSWORD = os.getenv('DB_PASSWORD') or env.get('DBPASSWORD')
DB_HOST = os.getenv('DB_HOST', 'aidbfree')
DB_PORT = os.getenv('DB_PORT', '1521')
DB_SERVICE = os.getenv('DB_SERVICE', 'FREEPDB1')
DB_DSN = os.getenv('DB_DSN', f'{DB_HOST}:{DB_PORT}/{DB_SERVICE}')

PRIVATEAI_BASE_URL = os.getenv('PRIVATEAI_BASE_URL', 'http://privateai:8080').rstrip('/')
PREFERRED_MODEL = os.getenv('PRIVATEAI_MODEL', 'all-minilm-l12-v2')

print('ENV file:', ENV_PATH if os.path.exists(ENV_PATH) else 'not found')
print('DB user:', DB_USER)
print('DB dsn :', DB_DSN)
print('Private AI URL:', PRIVATEAI_BASE_URL)
print('Preferred model:', PREFERRED_MODEL)

if not DB_PASSWORD:
    raise ValueError('DB password not found. Set DB_PASSWORD or DBPASSWORD in /home/.env')


## 2) Validate Private AI and choose a text-embedding model

In [ ]:
health = requests.get(f'{PRIVATEAI_BASE_URL}/health', timeout=20)
print('Health status:', health.status_code)
health.raise_for_status()

models_resp = requests.get(f'{PRIVATEAI_BASE_URL}/v1/models', timeout=20)
models_resp.raise_for_status()
models = models_resp.json().get('data', [])

if not models:
    raise RuntimeError('No models returned by /v1/models')

def norm(s):
    return (s or '').strip().lower()

text_models = []
for model in models:
    mid = model.get('id')
    caps = [c.upper() for c in model.get('modelCapabilities', [])]
    print(' -', mid, '|', ','.join(caps))
    if mid and any(c in ('TEXT_EMBEDDINGS', 'EMBEDDINGS') for c in caps):
        text_models.append(mid)

if not text_models:
    raise RuntimeError('No TEXT_EMBEDDINGS-capable models found')

MODEL_ID = next((m for m in text_models if norm(m) == norm(PREFERRED_MODEL)), None)
if MODEL_ID is None:
    MODEL_ID = next((m for m in text_models if norm(m) == 'all-minilm-l12-v2'), text_models[0])

print()
print('Selected model:', MODEL_ID)


## 3) Direct `/v1/embeddings` test

In [ ]:
payload = {
    'model': MODEL_ID,
    'input': [
        'Oracle AI Database supports vector similarity search.',
        'Private AI Services Container runs embedding models close to your data.'
    ],
}

resp = requests.post(f'{PRIVATEAI_BASE_URL}/v1/embeddings', json=payload, timeout=60)
if not resp.ok:
    print('Status :', resp.status_code)
    print('Body   :', resp.text[:1500])
resp.raise_for_status()

embed_json = resp.json()
first_vec = embed_json['data'][0]['embedding']
EMBEDDING_DIM = len(first_vec)

print('Returned vectors:', len(embed_json.get('data', [])))
print('Embedding dimension:', EMBEDDING_DIM)


## 4) Connect to Oracle Database

In [ ]:
conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
cur = conn.cursor()

cur.execute('select user from dual')
print('Connected as:', cur.fetchone()[0])


## 5) Create table and store Private AI embeddings

In [ ]:
TABLE_NAME = 'PRIVATEAI_DOCS_CONTAINER'

try:
    cur.execute(f'DROP TABLE {TABLE_NAME} PURGE')
except oracledb.DatabaseError:
    pass

cur.execute(f'''
    CREATE TABLE {TABLE_NAME} (
        doc_id NUMBER GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
        title VARCHAR2(200) NOT NULL,
        content CLOB NOT NULL,
        embedding VECTOR({EMBEDDING_DIM}, FLOAT32)
    )
''')

docs = [
    ('Why Private AI Container', 'Private AI keeps inference local and exposes embedding APIs by container endpoint.'),
    ('Why Vector Search', 'Vector search compares semantic meaning and not only exact keyword matches.'),
    ('JupyterLab Delivery', 'Notebooks are a practical way to prototype retrieval and embedding pipelines quickly.'),
    ('RAG Workflow', 'Chunk, embed, store, and retrieve relevant context at question time.'),
]

embed_params = json.dumps({
    'provider': 'privateai',
    'url': f'{PRIVATEAI_BASE_URL}/v1/embeddings',
    'host': 'local',
    'model': MODEL_ID,
})

inserted = 0
for title, content in docs:
    cur.execute(f'''
        INSERT INTO {TABLE_NAME} (title, content, embedding)
        VALUES (
            :title,
            :content,
            DBMS_VECTOR.UTL_TO_EMBEDDING(:content, JSON(:params))
        )
    ''', {'title': title, 'content': content, 'params': embed_params})
    inserted += 1

conn.commit()
print('Inserted rows:', inserted)


## 6) Similarity search

In [ ]:
query_text = 'How can I run embeddings locally and use them for semantic search?'

sql = f'''
SELECT
    title,
    ROUND(1 - VECTOR_DISTANCE(
        embedding,
        DBMS_VECTOR.UTL_TO_EMBEDDING(:query_text, JSON(:params)),
        COSINE
    ), 4) AS similarity,
    SUBSTR(content, 1, 120) AS preview
FROM {TABLE_NAME}
ORDER BY VECTOR_DISTANCE(
    embedding,
    DBMS_VECTOR.UTL_TO_EMBEDDING(:query_text, JSON(:params)),
    COSINE
)
FETCH FIRST 3 ROWS ONLY
'''

cur.execute(sql, {'query_text': query_text, 'params': embed_params})
rows = cur.fetchall()

print('Query:', query_text)
for idx, (title, score, preview) in enumerate(rows, 1):
    print(f'{idx}. {title} | score={score}')
    print(f'   {preview}')


In [ ]:
# Optional cleanup
# cur.execute(f'DROP TABLE {TABLE_NAME} PURGE')
# conn.commit()


In [ ]:
cur.close()
conn.close()
print('Connection closed.')
